In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data=pd.read_excel('../data/data.xlsx',sheet_name="Online Retail")

In [ ]:
print("SHape :",data.shape)
print("\nHead :",data.head())

In [ ]:
print("Data types:",data.dtypes)
print("\nDescription :")
print(data.describe(include="all"))

In [ ]:
missing=data.isnull().sum()

duplicates=data.duplicated().sum()

print("Missing values:\n",missing)

print("\nNumber of exact duplicates:", duplicates)

In [ ]:
cancelled = data["InvoiceNo"].astype(str).str.startswith("C")
print("Cancelled invoices:", cancelled.sum())

In [ ]:
invalid_quantity = data["Quantity"] <= 0

print("Quantity <= 0:", invalid_quantity.sum())

In [ ]:
invalid_price = data["UnitPrice"] <= 0

print("UnitPrice <= 0:", invalid_price.sum())

In [ ]:
anomalies=pd.DataFrame({
    "Anomaly":[
        "Cancelled Invoices",
        "Quantity<=0",
        "UnitPrice<=0"
    ],
    "Number of rows":[
        cancelled.sum(),
        invalid_price.sum(),
        invalid_quantity.sum()
    ]
})

print(anomalies)

In [8]:
data_clean=data.copy()

data_clean = data_clean.dropna(subset=["CustomerID"]) #if the customerID is nan drop the row

data_clean=data_clean.drop_duplicates()

data_clean = data_clean[
    ~data_clean["InvoiceNo"].astype(str).str.startswith("C")
]

data_clean = data_clean[data_clean["Quantity"] > 0]

data_clean = data_clean[data_clean["UnitPrice"] > 0]

data_clean["InvoiceDate"] = pd.to_datetime(data_clean["InvoiceDate"])

data_clean["TotalPrice"] = (
    data_clean["Quantity"] * data_clean["UnitPrice"]
)

print(data_clean.shape)
print(data_clean.head())

(392692, 9)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  TotalPrice  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom       15.30  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom       22.00  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  


In [ ]:
#Transactions by country
country_count=data_clean["Country"].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=country_count.values, y=country_count.index)
plt.title("Top 10 des pays par nombre de transactions")
plt.xlabel("Nombre de transactions")
plt.ylabel("Pays")
plt.show()

In [ ]:
#Transactions by month

monthly=(data_clean.set_index("InvoiceDate").resample("ME").size())
plt.figure(figsize=(10, 5))
monthly.plot()
plt.title("Nombre de transactions par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre de transactions")
plt.show()

In [ ]:
#Top products
top_products=(data_clean.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10))
plt.figure(figsize=(10, 5))
sns.barplot(x=top_products.values,y=top_products.index)
plt.title("Top 10 des produits les plus vendus")
plt.xlabel("Quantité vendue")
plt.ylabel("Produit")
plt.show()

| RFM | Meaning |
|---|---|
| **Recency** | How recently the customer purchased |
| **Frequency** | How often the customer purchased |
| **Monetary** | How much the customer spent |

In [ ]:
print("Shape:", data_clean.shape)

print("\nColumns:")
print(data_clean.columns.tolist())

print("\nData types:")
print(data_clean.dtypes)

print("\nMissing values:")
print(data_clean.isnull().sum())

In [ ]:
print("Number of transactions:", data_clean["InvoiceNo"].nunique())
print("Number of customers:", data_clean["CustomerID"].nunique())
print("\nDate range:")
print("First transaction:", data_clean["InvoiceDate"].min())
print("Last transaction:", data_clean["InvoiceDate"].max())

In [11]:
data_clean["TotalPrice"].describe()

count    392692.000000
mean         22.631500
std         311.099224
min           0.001000
25%           4.950000
50%          12.450000
75%          19.800000
max      168469.600000
Name: TotalPrice, dtype: float64

In [17]:
customer_activity = (
    data_clean.groupby("CustomerID")
    .agg(
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalPrice", "sum")
    )
    .sort_values(by="Monetary", ascending=False)
)
print(customer_activity.head(10))

            Frequency   Monetary
CustomerID                      
14646.0            73  280206.02
18102.0            60  259657.30
17450.0            46  194390.79
16446.0             2  168472.50
14911.0           201  143711.17
12415.0            21  124914.53
14156.0            55  117210.08
17511.0            31   91062.38
16029.0            63   80850.84
12346.0             1   77183.60


In [20]:
#recency
Last_date = data_clean["InvoiceDate"].max() + pd.Timedelta(days=1)
print(Last_date)

2011-12-10 12:50:00


In [22]:
customer_recency = data_clean.groupby("CustomerID")["InvoiceDate"].max()

In [23]:
customer_recency = (Last_date - customer_recency).dt.days

In [27]:
rfm = customer_activity.copy()
rfm["Recency"] = customer_recency
rfm.head(10)

,Frequency,Monetary,Recency
CustomerID,,,
14646.0,73,280206.02,2
18102.0,60,259657.30,1
17450.0,46,194390.79,8
16446.0,2,168472.50,1
14911.0,201,143711.17,1
12415.0,21,124914.53,24
14156.0,55,117210.08,10
17511.0,31,91062.38,3
16029.0,63,80850.84,39


In [34]:
#qcut divides your customers into groups based on their values
#5 divide the customers into 5 groups
rfm["R_score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1])
print(rfm.head(10))

            Frequency   Monetary  Recency R_score
CustomerID                                       
14646.0            73  280206.02        2       5
18102.0            60  259657.30        1       5
17450.0            46  194390.79        8       5
16446.0             2  168472.50        1       5
14911.0           201  143711.17        1       5
12415.0            21  124914.53       24       4
14156.0            55  117210.08       10       5
17511.0            31   91062.38        3       5
16029.0            63   80850.84       39       3
12346.0             1   77183.60      326       1


In [ ]:
rfm["F_score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)
print(rfm.head(10))

In [46]:
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5])
print(rfm.head(10))

            Frequency   Monetary  Recency R_score F_score M_score
CustomerID                                                       
14646.0            73  280206.02        2       5       5       5
18102.0            60  259657.30        1       5       5       5
17450.0            46  194390.79        8       5       5       5
16446.0             2  168472.50        1       5       2       5
14911.0           201  143711.17        1       5       5       5
12415.0            21  124914.53       24       4       5       5
14156.0            55  117210.08       10       5       5       5
17511.0            31   91062.38        3       5       5       5
16029.0            63   80850.84       39       3       5       5
12346.0             1   77183.60      326       1       1       5


In [47]:
rfm["RFM_Score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

print(rfm.head(10))

            Frequency   Monetary  Recency R_score F_score M_score RFM_Score
CustomerID                                                                 
14646.0            73  280206.02        2       5       5       5       555
18102.0            60  259657.30        1       5       5       5       555
17450.0            46  194390.79        8       5       5       5       555
16446.0             2  168472.50        1       5       2       5       525
14911.0           201  143711.17        1       5       5       5       555
12415.0            21  124914.53       24       4       5       5       455
14156.0            55  117210.08       10       5       5       5       555
17511.0            31   91062.38        3       5       5       5       555
16029.0            63   80850.84       39       3       5       5       355
12346.0             1   77183.60      326       1       1       5       115


In [48]:
def segment_customer(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    
    elif row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Loyal Customers"
    
    elif row["R_score"] >= 4:
        return "Recent Customers"
    
    elif row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    
    elif row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Lost Customers"
    
    else:
        return "Potential Customers"


In [49]:
rfm["Segment"]=rfm.apply(segment_customer,axis=1)
print(rfm.head(10))

            Frequency   Monetary  Recency R_score F_score M_score RFM_Score  \
CustomerID                                                                    
14646.0            73  280206.02        2       5       5       5       555   
18102.0            60  259657.30        1       5       5       5       555   
17450.0            46  194390.79        8       5       5       5       555   
16446.0             2  168472.50        1       5       2       5       525   
14911.0           201  143711.17        1       5       5       5       555   
12415.0            21  124914.53       24       4       5       5       455   
14156.0            55  117210.08       10       5       5       5       555   
17511.0            31   91062.38        3       5       5       5       555   
16029.0            63   80850.84       39       3       5       5       355   
12346.0             1   77183.60      326       1       1       5       115   

                        Segment  
CustomerID       